# R4-U1b (2/2): AST-A partial fine-tune, bundle and parity under Protocol A

Select a Colab **T4 GPU** runtime. Run this after the warm-up notebook (1/2). Set `WARMUP_ID` below to its `RUN_ID`. Sync the new code, the configs, the R4-S1 outputs, the warm-up run folder `runs/r4/<WARMUP_ID>`, the frozen manifests and the cohort shards to Drive first.

From the A warm-up it unfreezes the last two AST blocks, the final norm and the head (B baseline recipe: backbone 1e-5, head 1e-4, 8 epochs, effective batch 16, 5% warm-up then cosine, fp16). Expect about 1 to 1.5 hours on a T4. Before training it proves non-zero gradients and changed checksums only in the unfrozen groups and reproduces the warm-up's selection F1 at epoch 0; after training it checks the checksums again. The selection is frozen before the calibration and operating roles are scored. No test audio is read. The recipe's `status` must record the team's agreement.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys, time
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')  # Change if your project is elsewhere.
LEDGER = ROOT / 'reports/ledger.csv'
OWNER = 'Usman'  # Person running this notebook; written to the ledger.
MODEL_CACHE = '/content/hf_cache'  # The pinned AST is downloaded here and hash-verified; Colab-local, not synced.
SHARDS_DRIVE = ROOT / 'data/processed/cohort_v1'
SHARDS = Path('/content/shards')  # local copy of the shards: much faster to read than Drive
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
sys.path.insert(0, str(ROOT / 'src'))
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()

WARMUP_ID = ''  # REQUIRED: the RUN_ID printed by the warm-up notebook
assert WARMUP_ID, 'Set WARMUP_ID to the A warm-up run id'
WARMUP_DIR = ROOT / 'runs/r4' / WARMUP_ID
BUNDLE = ROOT / 'bundles/r4_u1_ast_partial_A'
# For a new run leave RUN_ID blank. To resume, enter the exact previous RUN_ID (code, environment and recipe must be unchanged).
RUN_ID = ''
RESUME = bool(RUN_ID)
if not RESUME:
    RUN_ID = 'r4_u1_ast_ft_A_' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
RUN_DIR = ROOT / 'runs/r4' / RUN_ID
print('Warm-up:', WARMUP_ID, '| Run:', RUN_ID, 'Resume:', RESUME, '| Run folder:', RUN_DIR)
assert (ROOT / 'scripts/r4_u1_ast.py').is_file(), 'Sync the updated project code to Drive'

In [ ]:
import torch, transformers, numpy
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, '| torch', torch.__version__,
      '| transformers', transformers.__version__, '| numpy', numpy.__version__)
assert torch.cuda.is_available(), 'Choose a T4 GPU runtime'
assert transformers.__version__ == '5.22.1', f'the B AST runs used transformers 5.22.1; found {transformers.__version__}'
lock = json.loads((ROOT / 'locks/B_lock_gr3.json').read_text(encoding='utf-8'))
contracts_sha = sha256(ROOT / 'src/echoproof/contracts.py')
assert contracts_sha in json.dumps(lock), f'src/echoproof/contracts.py differs from the GR3 lock ({contracts_sha[:12]}); refresh it from git'

required = ['configs/r4_u1_ast_warmup_a.json', 'configs/r4_u1_ast_ft_a.json', 'configs/r3_u2_ast_ft_b.json', 'configs/r2_u2_ast_b.json',
            'data/manifests/cohort_v1.parquet', 'data/splits/protocol_A_v1.parquet', 'data/processed/chunks_v1.parquet',
            'data/features/r4_s1_A/r4_s1_manifest.json', 'reports/r1/split_report.json', 'locks/B_lock_gr3.json',
            f'runs/r4/{WARMUP_ID}/warmup_result.json', f'runs/r4/{WARMUP_ID}/best.pt']
missing = [p for p in required if not (ROOT / p).is_file()]
assert not missing, f'Missing inputs: {missing}'
assert SHARDS_DRIVE.is_dir(), 'Set SHARDS_DRIVE to the canonical shard directory'
recipe = json.loads((ROOT / 'configs/r4_u1_ast_ft_a.json').read_text(encoding='utf-8'))
assert recipe['status'].startswith('team_agreed'), f"recipe status is {recipe['status']!r}: the team's agreement must be recorded first"
warm = json.loads((WARMUP_DIR / 'warmup_result.json').read_text(encoding='utf-8'))
assert warm['problems'] == [] and sha256(WARMUP_DIR / 'best.pt') == warm['best_checkpoint_sha256'], 'the warm-up result or checkpoint is not clean'
print('warm-up best epoch', warm['best_epoch'], '| selection macro F1', warm['best_macro_f1'])
contract = RUN_DIR.parent / (RUN_DIR.name + '_contract.json')
assert contract.is_file() if RESUME else not RUN_DIR.exists() and not contract.exists(), 'Check RUN_ID and the resume setting'
assert not BUNDLE.exists() or RESUME, 'BUNDLE already exists; choose a new folder'

## Fixture tests

In [ ]:
tests = ['tests/test_r4_u1_ast.py', 'tests/test_r4_u0_wavlm_cache.py', 'tests/test_r3_u2_ast.py', 'tests/test_r3_u2_adapter.py', 'tests/test_r1_u2_ast.py',
         'tests/test_m1_trainer.py']
check = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', *tests], text=True, capture_output=True)
print(check.stdout[-4000:], check.stderr[-2000:])
assert check.returncode == 0 and ' skipped' not in check.stdout, 'Resolve test failures or skips before training'

## Copy the shards to local disk

In [ ]:
import shutil
SHARDS.mkdir(exist_ok=True)
t0 = time.time()
for shard in sorted(SHARDS_DRIVE.glob('shard_*.tar')):
    target = SHARDS / shard.name
    if not target.exists():
        shutil.copyfile(shard, target)
    assert target.stat().st_size == shard.stat().st_size, shard.name
print(len(list(SHARDS.glob('shard_*.tar'))), 'shards on local disk,', round(time.time() - t0), 's')

## Fine-tune, select and export
Resume after a disconnect by setting `RUN_ID` above and rerunning from the top.

In [ ]:
cmd = [sys.executable, 'scripts/r4_u1_ast.py', 'finetune', '--root', str(ROOT), '--shard-dir', str(SHARDS), '--run-dir', str(RUN_DIR.relative_to(ROOT)),
       '--warmup-dir', str(WARMUP_DIR.relative_to(ROOT)), '--ledger', str(LEDGER.relative_to(ROOT)), '--owner', OWNER, '--model-cache', MODEL_CACHE]
if RESUME:
    cmd.append('--resume')
subprocess.run(cmd, check=True)

## Package the verified inference bundle
Converts the selected `best.pt` to safetensors with the pinned config (checked tensor by tensor; frozen groups must equal the pinned pretrained weights, unfrozen groups must differ). Uncalibrated: R4-N1 is still required.

In [ ]:
subprocess.run([sys.executable, 'scripts/r4_u1_ast.py', 'package', '--root', str(ROOT), '--run-dir', str(RUN_DIR.relative_to(ROOT)), '--owner', OWNER,
                '--bundle', str(BUNDLE.relative_to(ROOT)), '--model-cache', MODEL_CACHE], check=True)

## Adapter parity
Scores 50 selection chunks (all five components) from raw audio with the bundle adapter. The fp16 CUDA adapter must reproduce the exported probabilities within 0.01; fp32 on CUDA and CPU are measured and reported, not gated.

In [ ]:
subprocess.run([sys.executable, 'scripts/r4_u1_ast.py', 'parity', '--root', str(ROOT), '--run-dir', str(RUN_DIR.relative_to(ROOT)), '--owner', OWNER,
                '--shard-dir', str(SHARDS), '--bundle', str(BUNDLE.relative_to(ROOT))], check=True)

## Verify the handoff and write the report
Checks recorded hashes, the completed ledger row, the adaptation evidence, the epoch-0 reproduction, the frozen selection and the latest parity result. It does not replace independent review.

In [ ]:
evidence = json.loads((RUN_DIR / 'evidence.json').read_text(encoding='utf-8'))
assert evidence['status'] == 'validation_predictions_complete'
assert sha256(RUN_DIR / 'selection.json') == evidence['selection_sha256']
assert all(sha256(RUN_DIR / name) == digest for name, digest in evidence['output_sha256'].items())
selection = json.loads((RUN_DIR / 'selection.json').read_text(encoding='utf-8'))
assert sha256(RUN_DIR / 'best.pt') == selection['checkpoint_sha256']['baseline']
with LEDGER.open(newline='', encoding='utf-8') as f:
    rows = [r for r in csv.DictReader(f) if r['run_id'] == RUN_ID and r['status'] == 'completed']
assert len(rows) == 1, 'expected one completed ledger row for the fine-tune'
ev = json.loads((RUN_DIR / 'adaptation_evidence.json').read_text(encoding='utf-8'))
proof = json.loads((RUN_DIR / 'gradient_proof.json').read_text(encoding='utf-8'))
epoch0 = json.loads((RUN_DIR / 'epoch0_metrics.json').read_text(encoding='utf-8'))
assert ev['problems'] == [] and ev['frozen_groups_unchanged_final_and_best'] and proof['ok'] and epoch0['within_tolerance'] is True
assert ev['unfrozen_groups_changed_best'] == ev['trainable_groups'] and ev['epochs_run'] == 8
parity_path = sorted((ROOT / 'reports/r4').glob('r4_u1_ast_parity_*.json'))[-1]
parity = json.loads(parity_path.read_text(encoding='utf-8'))
assert parity['passed'] is True
print('selected epoch', selection['selected_epoch'], '| selection macro F1', selection['selection_macro_f1'], '| warm-up F1', warm['best_macro_f1'])
print('skipped GradScaler steps:', ev['skipped_steps'], '| parity fp16 max |dp|:', parity['fp16']['max_abs_difference'])
report = {'task': 'R4-U1b', 'run_id': RUN_ID, 'warmup_id': WARMUP_ID, 'run_dir': str(RUN_DIR), 'bundle': str(BUNDLE),
          'evidence_sha256': sha256(RUN_DIR / 'evidence.json'), 'selection_sha256': evidence['selection_sha256'],
          'adaptation_evidence_sha256': sha256(RUN_DIR / 'adaptation_evidence.json'), 'warmup_result_sha256': sha256(WARMUP_DIR / 'warmup_result.json'),
          'parity_report': parity_path.name, 'parity_report_sha256': sha256(parity_path), 'ledger_rows': len(rows),
          'selected_epoch': selection['selected_epoch'], 'selection_macro_f1': selection['selection_macro_f1'], 'counts': evidence['counts'],
          'selection_panels': evidence['selection_panels'], 'paired': {k: evidence['paired_warmup_vs_adapted'][k] for k in ('recordings', 'decision_changes_at_0.5', 'macro_f1_change')},
          'torch': torch.__version__, 'transformers': transformers.__version__, 'gpu': torch.cuda.get_device_name(0)}
path = ROOT / 'reports/r4' / ('r4_u1_ast_colab_' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()) + '.json')
assert not path.exists()
path.write_text(json.dumps(report, indent=2, sort_keys=True), encoding='utf-8')
print('report:', path)